In [1]:
# mini_sissoをsrc/配下から探す
import sys
import os

sys.path.append(os.path.join(os.getcwd(), "src"))

In [3]:
# test_mini_sisso.py

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

# 完成したMiniSissoクラスをインポート
# (mini_sissoというパッケージ名でインストールされているか、
#  あるいは同じディレクトリ構造にあることを前提とします)
from mini_sisso.model import MiniSisso

# --- 0. 共通のテスト用データを準備 ---
np.random.seed(42)
X_np = np.random.rand(200, 2)
X_np[:, 0] *= 2  # f0の範囲を[0, 2]に
X_np[:, 1] *= 3  # f1の範囲を[0, 3]に

# 真の式: y = 2 * sin(f0) + f1^2 + ノイズ
y_np = 2 * np.sin(X_np[:, 0]) + X_np[:, 1] ** 2 + np.random.randn(200) * 0.1

X_df = pd.DataFrame(X_np, columns=["feature_A", "feature_B"])
y_series = pd.Series(y_np, name="target")

X_train_np, X_test_np, y_train, y_test = train_test_split(X_np, y_np, test_size=0.3, random_state=42)
X_train_df, X_test_df, _, _ = train_test_split(X_df, y_np, test_size=0.3, random_state=42)


print("=" * 60)
print(" Comprehensive Test for mini-sisso Library")
print("=" * 60)


# --- 1. 基本機能テスト (fit/predict) ---
print("\n--- [Test 1] Basic Functionality: fit() and predict() ---")
try:
    # モデルのインスタンス化
    model = MiniSisso(
        n_expansion=2,
        n_term=2,
        operators=["+", "sin", "pow2"],
    )

    # NumPy配列で学習
    print("\nFitting with NumPy arrays...")
    model.fit(X_train_np, y_train)

    # Pandas DataFrameで予測
    predictions = model.predict(X_test_df)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    print(f"\nSUCCESS: Basic fit/predict test completed.")
    print(f"  - Discovered Equation: {model.equation_}")
    print(f"  - RMSE on test data: {rmse:.4f}")
    assert rmse < 0.2, "Basic test RMSE is too high!"

except Exception as e:
    print(f"\nFAILED: Basic functionality test encountered an error: {e}")

# --- 2. scikit-learn互換性テスト ---
print("\n--- [Test 2] scikit-learn Compatibility ---")
try:
    model = MiniSisso(n_term=2, operators=["+", "sin", "pow2"])

    # (A) .score() メソッドのテスト
    print("\n(A) Testing .score() method (calculates R2 score)...")
    model.fit(X_train_df, y_train)
    r2_score = model.score(X_test_df, y_test)
    print(f"  - .score() on test data (R2): {r2_score:.4f}")
    assert r2_score > 0.9, ".score() R2 is too low!"

    # (B) get_params/set_params のテスト
    print("\n(B) Testing get_params() and set_params()...")
    params = model.get_params()
    print(f"  - get_params() output: {params['n_term']}")
    assert params["n_term"] == 2, "get_params() failed."

    model.set_params(n_term=1)
    print(f"  - set_params(n_term=1) executed.")
    assert model.get_params()["n_term"] == 1, "set_params() failed."
    print("  - SUCCESS: get_params/set_params work as expected.")

except Exception as e:
    print(f"\nFAILED: scikit-learn compatibility test encountered an error: {e}")

# --- 3. scikit-learnツール群との連携テスト ---
print("\n--- [Test 3] Integration with scikit-learn Tools ---")
try:
    # (A) Pipelineとの連携テスト
    print("\n(A) Testing integration with Pipeline...")
    pipeline = Pipeline([("scaler", StandardScaler()), ("sisso", MiniSisso(n_expansion=2, n_term=2, operators=["+", "sin", "pow2"]))])
    pipeline.fit(X_train_np, y_train)
    pipeline_predictions = pipeline.predict(X_test_np)
    pipeline_rmse = np.sqrt(mean_squared_error(y_test, pipeline_predictions))
    print(f"  - SUCCESS: Pipeline fit/predict completed.")
    print(f"  - RMSE from pipeline: {pipeline_rmse:.4f}")
    assert pipeline_rmse < 0.2, "Pipeline RMSE is too high!"

    # (B) GridSearchCVとの連携テスト
    print("\n(B) Testing integration with GridSearchCV...")
    param_grid = {"n_term": [1, 2], "k_per_level": [20, 30]}

    # MiniSissoのインスタンスを作成
    sisso_for_grid = MiniSisso(n_expansion=2, operators=["+", "sin", "pow2"])

    # GridSearchCVを実行
    grid_search = GridSearchCV(sisso_for_grid, param_grid, cv=2, scoring="neg_root_mean_squared_error")  # テスト時間を短縮するためcv=2

    print("  - Running GridSearchCV (this may take a moment)...")
    grid_search.fit(X_np, y_np)

    print(f"  - SUCCESS: GridSearchCV completed.")
    print(f"  - Best parameters found: {grid_search.best_params_}")
    print(f"  - Best cross-validation RMSE: {-grid_search.best_score_:.4f}")
    assert "n_term" in grid_search.best_params_, "GridSearchCV failed to find best params."

except Exception as e:
    print(f"\nFAILED: scikit-learn integration test encountered an error: {e}")

print("\n" + "=" * 60)
print(" All tests completed.")
print("=" * 60)

 Comprehensive Test for mini-sisso Library

--- [Test 1] Basic Functionality: fit() and predict() ---

Fitting with NumPy arrays...
*** Starting Level-wise Recipe Generation (Level-wise SIS: ON, k_per_level=50) ***
Level 1: Generated 5, selected top 5. Total promising: 7. Time: 0.00s
Level 2: Generated 30, selected top 30. Total promising: 37. Time: 0.00s

===== Searching for 1-term models =====
SIS selected 10 new features. Pool size: 10
--- Running SO for 1-term models. Total combinations: 10 ---
Best 1-term model: RMSE=0.250981, Eq: +0.988518 * (f0 + ^2(f1)) +0.455044
Time: 0.01 seconds

===== Searching for 2-term models =====
SIS selected 10 new features. Pool size: 20
--- Running SO for 2-term models. Total combinations: 190 ---
Best 2-term model: RMSE=0.088097, Eq: +0.997666 * ^2(f1) +2.014116 * sin(f0) +0.001270
Time: 0.01 seconds

SISSO fitting finished. Total time: 0.02s

Best Model Found (2 terms):
  RMSE: 0.088097
  R2:   0.998927
  Equation: +0.997666 * ^2(f1) +2.014116 * s

In [4]:
# test_mini_sisso.py (Corrected Pipeline Test)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

# 完成したMiniSissoクラスをインポート
from mini_sisso.model import MiniSisso

# --- 0. 共通のテスト用データを準備 ---
np.random.seed(42)
X_np = np.random.rand(200, 2)
X_np[:, 0] *= 2
X_np[:, 1] *= 3
y_np = 2 * np.sin(X_np[:, 0]) + X_np[:, 1] ** 2 + np.random.randn(200) * 0.1
X_df = pd.DataFrame(X_np, columns=["feature_A", "feature_B"])
X_train_np, X_test_np, y_train, y_test = train_test_split(X_np, y_np, test_size=0.3, random_state=42)
X_train_df, X_test_df, _, _ = train_test_split(X_df, y_np, test_size=0.3, random_state=42)

print("=" * 60)
print(" Comprehensive Test for mini-sisso Library")
print("=" * 60)

# --- 1. 基本機能テスト (fit/predict) ---
print("\n--- [Test 1] Basic Functionality: fit() and predict() ---")
try:
    model = MiniSisso(n_expansion=2, n_term=2, operators=["+", "sin", "pow2"])
    print("\nFitting with NumPy arrays...")
    model.fit(X_train_np, y_train)
    predictions = model.predict(X_test_df)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    print(f"\nSUCCESS: Basic fit/predict test completed.")
    print(f"  - Discovered Equation: {model.equation_}")
    print(f"  - RMSE on test data: {rmse:.4f}")
    assert rmse < 0.2, f"Basic test RMSE ({rmse}) is too high!"
except Exception as e:
    print(f"\nFAILED: Basic functionality test encountered an error: {e}")

# --- 2. scikit-learn互換性テスト ---
print("\n--- [Test 2] scikit-learn Compatibility ---")
try:
    model = MiniSisso(n_term=2, operators=["+", "sin", "pow2"])
    print("\n(A) Testing .score() method (calculates R2 score)...")
    model.fit(X_train_df, y_train)
    r2_score_val = model.score(X_test_df, y_test)
    print(f"  - .score() on test data (R2): {r2_score_val:.4f}")
    assert r2_score_val > 0.9, f".score() R2 ({r2_score_val}) is too low!"
    print("\n(B) Testing get_params() and set_params()...")
    params = model.get_params()
    assert params["n_term"] == 2, "get_params() failed."
    model.set_params(n_term=1)
    assert model.get_params()["n_term"] == 1, "set_params() failed."
    print("  - SUCCESS: get_params/set_params work as expected.")
except Exception as e:
    print(f"\nFAILED: scikit-learn compatibility test encountered an error: {e}")

# --- 3. scikit-learnツール群との連携テスト ---
print("\n--- [Test 3] Integration with scikit-learn Tools ---")
try:
    # (A) Pipelineとの連携テスト
    print("\n(A) Testing integration with Pipeline...")

    # ★★★ CORRECTION ★★★
    # StandardScalerはシンボリック回帰の性能を悪化させるため、パイプラインからは削除。
    # ここでは、MiniSissoがPipelineのステップとして技術的に機能することを確認する。
    pipeline = Pipeline([("sisso", MiniSisso(n_expansion=2, n_term=2, operators=["+", "sin", "pow2"]))])
    pipeline.fit(X_train_np, y_train)
    pipeline_predictions = pipeline.predict(X_test_np)
    pipeline_rmse = np.sqrt(mean_squared_error(y_test, pipeline_predictions))

    print(f"  - SUCCESS: Pipeline fit/predict completed.")
    print(f"  - RMSE from pipeline: {pipeline_rmse:.4f}")
    assert pipeline_rmse < 0.2, f"Pipeline RMSE ({pipeline_rmse}) is too high!"

    # (B) GridSearchCVとの連携テスト
    print("\n(B) Testing integration with GridSearchCV...")
    param_grid = {"n_term": [1, 2], "k_per_level": [20, 30]}
    grid_search = GridSearchCV(MiniSisso(n_expansion=2, operators=["+", "sin", "pow2"]), param_grid, cv=2, scoring="neg_root_mean_squared_error")
    print("  - Running GridSearchCV (this may take a moment)...")
    grid_search.fit(X_np, y_np)

    print(f"  - SUCCESS: GridSearchCV completed.")
    print(f"  - Best parameters found: {grid_search.best_params_}")
    print(f"  - Best cross-validation RMSE: {-grid_search.best_score_:.4f}")
    assert "n_term" in grid_search.best_params_, "GridSearchCV failed to find best params."

except Exception as e:
    print(f"\nFAILED: scikit-learn integration test encountered an error: {e}")

print("\n" + "=" * 60)
print(" All tests completed.")
print("=" * 60)

 Comprehensive Test for mini-sisso Library

--- [Test 1] Basic Functionality: fit() and predict() ---

Fitting with NumPy arrays...
*** Starting Level-wise Recipe Generation (Level-wise SIS: ON, k_per_level=50) ***
Level 1: Generated 5, selected top 5. Total promising: 7. Time: 0.00s
Level 2: Generated 30, selected top 30. Total promising: 37. Time: 0.00s

===== Searching for 1-term models =====
SIS selected 10 new features. Pool size: 10
--- Running SO for 1-term models. Total combinations: 10 ---
Best 1-term model: RMSE=0.250981, Eq: +0.988518 * (f0 + ^2(f1)) +0.455044
Time: 0.01 seconds

===== Searching for 2-term models =====
SIS selected 10 new features. Pool size: 20
--- Running SO for 2-term models. Total combinations: 190 ---
Best 2-term model: RMSE=0.088097, Eq: +0.997666 * ^2(f1) +2.014116 * sin(f0) +0.001270
Time: 0.01 seconds

SISSO fitting finished. Total time: 0.02s

Best Model Found (2 terms):
  RMSE: 0.088097
  R2:   0.998927
  Equation: +0.997666 * ^2(f1) +2.014116 * s